# Parquet Validation Notebook
This notebook validates Parquet files stored in MinIO from Kafka order-events topic

In [1]:
import pandas as pd
import pyarrow.parquet as pq
import s3fs
import os

# S3/MinIO configuration
s3 = s3fs.S3FileSystem(
    key='minioadmin',
    secret='minioadmin',
    client_kwargs={'endpoint_url': 'http://minio:9000'}
)

In [3]:
# List all files in datalake bucket
files = s3.ls('datalake/topics/order-events/calc_id=20251224-180000/dt=2025-12-24/hour=18/', detail=True)
print(f"Found {len(files)} objects in datalake bucket")
for f in files[:10]:  # Show first 10
    print(f"{f['Key']}: {f['Size'] / 1024 / 1024:.2f} MB")

Found 144 objects in datalake bucket
datalake/topics/order-events/calc_id=20251224-180000/dt=2025-12-24/hour=18/order-events+0+0000000000.snappy.parquet: 0.10 MB
datalake/topics/order-events/calc_id=20251224-180000/dt=2025-12-24/hour=18/order-events+0+0000013976.snappy.parquet: 0.10 MB
datalake/topics/order-events/calc_id=20251224-180000/dt=2025-12-24/hour=18/order-events+1+0000000000.snappy.parquet: 0.10 MB
datalake/topics/order-events/calc_id=20251224-180000/dt=2025-12-24/hour=18/order-events+1+0000013745.snappy.parquet: 0.10 MB
datalake/topics/order-events/calc_id=20251224-180000/dt=2025-12-24/hour=18/order-events+10+0000000000.snappy.parquet: 0.10 MB
datalake/topics/order-events/calc_id=20251224-180000/dt=2025-12-24/hour=18/order-events+10+0000013728.snappy.parquet: 0.10 MB
datalake/topics/order-events/calc_id=20251224-180000/dt=2025-12-24/hour=18/order-events+11+0000000000.snappy.parquet: 0.10 MB
datalake/topics/order-events/calc_id=20251224-180000/dt=2025-12-24/hour=18/order-even

In [23]:
# Read first Parquet file
parquet_files = [f['Key'] for f in files if f['Key'].endswith('.parquet')]
if parquet_files:
    first_file = f's3://{parquet_files[0]}'
    df = pd.read_parquet(first_file, filesystem=s3)
    print(f"Schema: {df.dtypes}")
    print(f"\nFirst 5 rows:\n{df.head()}")
    print(f"\nTotal rows: {len(df)}")

Schema: order_id                  object
customer_id               object
order_date                object
delivery_date             object
status                    object
total_amount             float64
currency                  object
item_count                 int32
shipping_address          object
billing_address           object
shipping_zip              object
billing_zip               object
shipping_city             object
billing_city              object
shipping_country          object
billing_country           object
payment_method            object
card_last_digits          object
card_expiry               object
ip_address                object
user_agent                object
campaign_id               object
referrer_url              object
device_type               object
browser                   object
os                        object
coupon_code               object
discount_amount          float64
loyalty_points_used        int32
gift_wrap                   bool
sp

In [24]:
# Check Parquet metadata
if parquet_files:
    with s3.open(parquet_files[0], 'rb') as f:
        parquet_file = pq.ParquetFile(f)
        print(f"Schema:\n{parquet_file.schema}")
        print(f"\nMetadata:\n{parquet_file.metadata}")
        print(f"\nNum row groups: {parquet_file.num_row_groups}")

Schema:
required group field_id=-1 ru.pospelov.etl.avro.OrderEvent {
  required binary field_id=-1 order_id (String);
  required binary field_id=-1 customer_id (String);
  required binary field_id=-1 order_date (String);
  optional int32 field_id=-1 delivery_date (Date);
  required binary field_id=-1 status (String);
  required double field_id=-1 total_amount;
  required binary field_id=-1 currency (String);
  required int32 field_id=-1 item_count;
  required binary field_id=-1 shipping_address (String);
  required binary field_id=-1 billing_address (String);
  required binary field_id=-1 shipping_zip (String);
  required binary field_id=-1 billing_zip (String);
  required binary field_id=-1 shipping_city (String);
  required binary field_id=-1 billing_city (String);
  required binary field_id=-1 shipping_country (String);
  required binary field_id=-1 billing_country (String);
  required binary field_id=-1 payment_method (String);
  required binary field_id=-1 card_last_digits (String

In [25]:
# Count total rows across all parquet files using metadata (fast, without loading data)
parquet_files = [f['Key'] for f in files if f['Key'].endswith('.parquet')]
total_rows = 0

print(f"Analyzing {len(parquet_files)} parquet files...\n")

for file_key in parquet_files:
  with s3.open(file_key, 'rb') as f:
      parquet_file = pq.ParquetFile(f)
      num_rows = parquet_file.metadata.num_rows
      total_rows += num_rows

print(f"{'='*60}")
print(f"Total files: {len(parquet_files)}")
print(f"Total rows: {total_rows:,}")


Analyzing 72 parquet files...

Total files: 72
Total rows: 1,000,000


In [30]:
# Проверить все файлы и партиции
files = s3.ls('datalake/topics/order-events/', detail=True, recursive=True)
parquet_files = [f for f in files if f['Key'].endswith('.parquet')]

print(f"Total Parquet files: {len(parquet_files)}\n")

# Группировка по партициям
from collections import defaultdict
partitions = defaultdict(list)

for f in parquet_files:
  # Извлечь путь партиции
  path_parts = f['Key'].split('/')
  if len(path_parts) >= 5:
      partition = '/'.join(path_parts[2:5])  # calc_id/dt/hour
      partitions[partition].append(f)

# Показать статистику по партициям
for partition, files_list in partitions.items():
  total_rows = 0
  for file_info in files_list:
      with s3.open(file_info['Key'], 'rb') as f:
          parquet_file = pq.ParquetFile(f)
          total_rows += parquet_file.metadata.num_rows
  print(f"{partition}: {len(files_list)} files, {total_rows:,} rows")

TypeError: S3FileSystem._ls() got an unexpected keyword argument 'recursive'

In [34]:
# Проверить все партиции
s3.invalidate_cache()
all_parquet_files = s3.glob('datalake/topics/order-events/**/*.parquet')

print(f"Total Parquet files: {len(all_parquet_files)}\n")

# Группировка по партициям
from collections import defaultdict
partitions = defaultdict(list)

for file_path in all_parquet_files:
  # Убедимся, что путь начинается с datalake/
  if not file_path.startswith('datalake/'):
      file_path = 'datalake/' + file_path

  path_parts = file_path.split('/')
  if len(path_parts) >= 5:
      partition = '/'.join(path_parts[2:5])
      partitions[partition].append(file_path)

# Показать статистику
print("Статистика по партициям:\n")
grand_total = 0

for partition, files_list in sorted(partitions.items()):
  partition_total = 0
  for file_path in files_list:
      try:
          with s3.open(file_path, 'rb') as f:
              pf = pq.ParquetFile(f)
              partition_total += pf.metadata.num_rows
      except Exception as e:
          print(f"Ошибка чтения {file_path}: {e}")
          continue

  grand_total += partition_total
  print(f"{partition}: {len(files_list)} files, {partition_total:,} rows")

print(f"\n{'='*60}")
print(f"TOTAL: {grand_total:,} rows")

Total Parquet files: 144

Статистика по партициям:

order-events/calc_id=20251224-180000/dt=2025-12-24: 144 files, 2,000,000 rows

TOTAL: 2,000,000 rows
